## Cálculo dos KPIs do tracking mobile: cleaned para curated
Lê `workspace.cleaned.tracking_events` e calcula os KPIs de uso do app (atividade diária, telas, API, ações de toque e sessões), salvando na camada `curated`. Este notebook não depende do Neon.

#### Configuração

##### Imports e camadas

In [0]:
from pyspark.sql import functions as F

CATALOG = "workspace"
CLEANED = f"{CATALOG}.cleaned"
CURATED = f"{CATALOG}.curated"
TIMEZONE = "America/Sao_Paulo"      
HTTP_ERRO_MIN = 400               

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CURATED}")

#### Leitura e gravação

##### Eventos do tracking

In [0]:
def read_events():
    return (spark.table(f"{CLEANED}.tracking_events")
            .filter(F.col("company_id").isNotNull())
            .withColumn("event_date", F.to_date(F.from_utc_timestamp("timestamp", TIMEZONE))))

##### Salvar uma tabela na curated

In [0]:
def save_curated(df, table):
    df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CURATED}.{table}")

##### Contar linhas que atendem a uma condição

In [0]:
def count_if(condition):
    return F.sum(F.when(condition, 1).otherwise(0))

#### Atividade

##### Atividade diária por empresa

In [0]:
def atividade_diaria_empresa(events):
    return events.groupBy("company_id", "event_date").agg(
        F.count("*").alias("eventos"),
        F.countDistinct("session_id").alias("sessoes"),
        F.countDistinct("user_id").alias("usuarios_ativos"),
        count_if(F.col("event_name") == "app_opened").alias("aberturas_app"),
    )

##### Sessões

In [0]:
def sessoes(events):
    df = events.groupBy("session_id", "company_id", "user_id").agg(
        F.min("timestamp").alias("inicio"),
        F.max("timestamp").alias("fim"),
        F.count("*").alias("eventos"),
        F.countDistinct("screen").alias("telas_distintas"),
    )
    return df.withColumn("duracao_s", F.round(F.col("fim").cast("double") - F.col("inicio").cast("double"), 1))

#### Telas e ações

##### Uso por tela

In [0]:
def uso_tela(events):
    views = (events.filter(F.col("event_name") == "screen_viewed")
                   .groupBy("company_id", "screen").agg(F.count("*").alias("visualizacoes")))
    tempo = (events.filter((F.col("event_name") == "screen_exited") & F.col("properties_duration_ms").isNotNull())
                   .groupBy("company_id", "screen").agg(
                       F.round(F.avg("properties_duration_ms"), 0).alias("tempo_medio_ms"),
                       F.percentile_approx("properties_duration_ms", 0.5).alias("tempo_mediano_ms")))
    return views.join(tempo, ["company_id", "screen"], "outer")

##### Ações de toque

In [0]:
def acoes(events):
    return (events.filter(F.col("event_name") == "ui_action")
                  .groupBy("company_id", "screen", "properties_target", "properties_input_method")
                  .agg(F.count("*").alias("acoes"))
                  .withColumnRenamed("properties_target", "alvo")
                  .withColumnRenamed("properties_input_method", "forma_entrada"))

#### API

##### Chamadas de API por dia e endpoint

In [0]:
def api_diaria(events):
    api = events.filter(F.col("event_name") == "api_request_completed")
    df = api.groupBy("company_id", "event_date", "properties_endpoint").agg(
        F.count("*").alias("requisicoes"),
        count_if(F.col("properties_http_status") >= HTTP_ERRO_MIN).alias("erros"),
        F.round(F.avg("properties_duration_ms"), 0).alias("latencia_media_ms"),
        F.percentile_approx("properties_duration_ms", 0.95).alias("latencia_p95_ms"),
    ).withColumnRenamed("properties_endpoint", "endpoint")
    return df.withColumn("taxa_erro_pct", F.round(100 * F.col("erros") / F.col("requisicoes"), 2))

#### Execução

##### Calcular os KPIs

In [0]:
events = read_events()

tabelas = {
    "kpi_tracking_diario_empresa": atividade_diaria_empresa(events),
    "kpi_tracking_sessao": sessoes(events),
    "kpi_tracking_tela": uso_tela(events),
    "kpi_tracking_acoes": acoes(events),
    "kpi_tracking_api_diaria": api_diaria(events),
}

##### Conferir o resultado

In [0]:
display(tabelas["kpi_tracking_diario_empresa"].orderBy("event_date"))

##### Salvar na curated

In [0]:
for nome, df in tabelas.items():
    save_curated(df, nome)
    print(nome, "ok")